In [1]:
from pathlib import Path
import json

import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

from data_utils import load_data_for_n, get_fold_split, N_LEVELS

MODELS_DIR = Path("../Models/Linear_baseline")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def evaluate_predictions(seg_preds, y_test, speaker_test):
    """Given per-segment predictions, compute both segment-level and
    recording-level (majority vote) accuracy and F1."""
    # segment-level
    seg_acc = (seg_preds == y_test).mean()
    seg_f1 = f1_score(y_test, seg_preds)

    # recording-level (majority vote per speaker)
    rec_true, rec_pred = [], []
    for spk in np.unique(speaker_test):
        mask = (speaker_test == spk)
        true_label = y_test[mask][0]
        vote = seg_preds[mask].mean() > 0.5
        rec_true.append(true_label)
        rec_pred.append(int(vote))

    rec_true, rec_pred = np.array(rec_true), np.array(rec_pred)
    rec_acc = (rec_pred == rec_true).mean()
    rec_f1 = f1_score(rec_true, rec_pred)

    return seg_acc, seg_f1, rec_acc, rec_f1

In [3]:
def run_cv_sklearn(model_factory, n: int, verbose=False):
    """
    Run 5-fold CV at N-level n using a scikit-learn classifier.

    model_factory: callable returning a fresh classifier each time.
                   E.g. lambda: LogisticRegression(C=1.0, max_iter=1000)
    """
    X, y, fold, speaker = load_data_for_n(n)

    seg_accs, seg_f1s, rec_accs, rec_f1s = [], [], [], []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(
            X, y, fold, speaker, fold_number
        )

        clf = model_factory()
        clf.fit(X_train, y_train)
        seg_preds = clf.predict(X_test).astype(int)

        s_acc, s_f1, r_acc, r_f1 = evaluate_predictions(seg_preds, y_test, speaker_test)
        seg_accs.append(s_acc); seg_f1s.append(s_f1)
        rec_accs.append(r_acc); rec_f1s.append(r_f1)

        if verbose:
            print(f"  Fold {fold_number}: rec_acc={r_acc:.4f}, rec_f1={r_f1:.4f}")

    return {
        "seg_acc_mean": np.mean(seg_accs), "seg_acc_std": np.std(seg_accs),
        "seg_f1_mean": np.mean(seg_f1s),  "seg_f1_std": np.std(seg_f1s),
        "rec_acc_mean": np.mean(rec_accs), "rec_acc_std": np.std(rec_accs),
        "rec_f1_mean": np.mean(rec_f1s),   "rec_f1_std": np.std(rec_f1s),
    }

In [4]:
result = run_cv_sklearn(
    model_factory=lambda: LogisticRegression(C=1.0, max_iter=1000),
    n=8,
    verbose=True,
)
print(f"\nN=8 LR — Recording-level: Acc={result['rec_acc_mean']:.4f}±{result['rec_acc_std']:.4f}, "
      f"F1={result['rec_f1_mean']:.4f}±{result['rec_f1_std']:.4f}")

  Fold 1: rec_acc=0.8333, rec_f1=0.8462
  Fold 2: rec_acc=0.9565, rec_f1=0.9600
  Fold 3: rec_acc=0.9565, rec_f1=0.9524
  Fold 4: rec_acc=0.7826, rec_f1=0.8387
  Fold 5: rec_acc=0.9565, rec_f1=0.9524

N=8 LR — Recording-level: Acc=0.8971±0.0745, F1=0.9099±0.0552


In [5]:
results_lr = {}
results_svm = {}

for n in tqdm(N_LEVELS, desc="N-levels"):
    print(f"\n=== N={n} ===")
    results_lr[n] = run_cv_sklearn(
        lambda: LogisticRegression(C=1.0, max_iter=1000), n=n
    )
    results_svm[n] = run_cv_sklearn(
        lambda: LinearSVC(C=1.0, max_iter=5000, dual="auto"), n=n
    )
    print(f"  LR:  Acc={results_lr[n]['rec_acc_mean']:.4f}±{results_lr[n]['rec_acc_std']:.4f}, "
          f"F1={results_lr[n]['rec_f1_mean']:.4f}±{results_lr[n]['rec_f1_std']:.4f}")
    print(f"  SVM: Acc={results_svm[n]['rec_acc_mean']:.4f}±{results_svm[n]['rec_acc_std']:.4f}, "
          f"F1={results_svm[n]['rec_f1_mean']:.4f}±{results_svm[n]['rec_f1_std']:.4f}")

# Save both, keyed cleanly
with open(MODELS_DIR / "results_summary.json", "w") as f:
    json.dump({
        "logistic_regression": {str(n): r for n, r in results_lr.items()},
        "linear_svm":          {str(n): r for n, r in results_svm.items()},
    }, f, indent=2)

print("\nSaved to results_summary.json")

N-levels:   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


N-levels:  14%|█▍        | 1/7 [00:13<01:19, 13.27s/it]

  LR:  Acc=0.9054±0.0743, F1=0.9193±0.0560
  SVM: Acc=0.9141±0.0542, F1=0.9255±0.0479

=== N=2 ===


N-levels:  29%|██▊       | 2/7 [00:31<01:21, 16.28s/it]

...

  LR:  Acc=0.8880±0.0804, F1=0.9003±0.0569
  SVM: Acc=0.8790±0.1009, F1=0.8898±0.0798

=== N=32 ===


N-levels:  86%|████████▌ | 6/7 [02:14<00:26, 26.07s/it]

  LR:  Acc=0.8533±0.0809, F1=0.8650±0.0556
  SVM: Acc=0.8620±0.0697, F1=0.8747±0.0503

=== N=64 ===


N-levels: 100%|██████████| 7/7 [03:00<00:00, 25.80s/it]

  LR:  Acc=0.8109±0.0789, F1=0.8232±0.0595
  SVM: Acc=0.8109±0.0789, F1=0.8232±0.0595

Saved to results_summary.json


In [6]:
results_lr_sweep = {C: {} for C in [1.0, 0.1, 0.01]}

for C in [1.0, 0.1, 0.01]:
    print(f"\n=== C = {C} ===")
    for n in tqdm(N_LEVELS, desc=f"C={C}", leave=False):
        results_lr_sweep[C][n] = run_cv_sklearn(
            lambda: LogisticRegression(C=C, max_iter=1000), n=n
        )

# Print comparison table
print(f"\n{'N':>3} | {'C=1.0':>18} | {'C=0.1':>18} | {'C=0.01':>18}")
print("-" * 70)
for n in N_LEVELS:
    row = f"{n:>3} |"
    for C in [1.0, 0.1, 0.01]:
        r = results_lr_sweep[C][n]
        row += f" {r['rec_f1_mean']*100:5.2f}% ± {r['rec_f1_std']*100:4.2f}% |"
    print(row)


=== C = 1.0 ===



=== C = 0.1 ===



=== C = 0.01 ===



  N |              C=1.0 |              C=0.1 |             C=0.01
----------------------------------------------------------------------
  1 | 91.93% ± 5.60% | 88.86% ± 5.50% | 77.66% ± 7.91% |
  2 | 90.30% ± 3.03% | 90.99% ± 5.52% | 83.72% ± 3.63% |
  4 | 90.89% ± 5.76% | 89.35% ± 6.18% | 88.15% ± 7.35% |
  8 | 90.99% ± 5.52% | 92.67% ± 5.71% | 91.94% ± 6.50% |
 16 | 90.03% ± 5.69% | 89.48% ± 8.46% | 89.72% ± 9.33% |
 32 | 86.50% ± 5.56% | 87.72% ± 9.06% | 83.89% ± 9.34% |
 64 | 82.32% ± 5.95% | 82.37% ± 5.16% | 84.27% ± 6.69% |
